In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# PyTorch Fundamentals — tensor đến training step

Mục tiêu: kiểm tra shape/device/dtype, dùng Dataset/DataLoader và hoàn thành một training step có gradient.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

x = torch.tensor([[1., 2.], [3., 4.]])
assert x.shape == (2, 2) and x.dtype == torch.float32
device = torch.device("cuda" if torch.cuda.is_available() and not FAST_MODE else "cpu")
x = x.to(device)
print("shape/dtype/device:", x.shape, x.dtype, x.device)

In [ ]:
features = torch.linspace(-1, 1, 64).unsqueeze(1)
targets = 3 * features - 0.5
loader = DataLoader(TensorDataset(features, targets), batch_size=16, shuffle=False)
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
loss_fn = torch.nn.MSELoss()

initial = loss_fn(model(features), targets).item()
for _ in range(8 if FAST_MODE else 40):
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
final = loss_fn(model(features), targets).item()
assert final < initial
print(f"loss: {initial:.4f} -> {final:.4f}")

## Diagnose

- `loss.backward()` chỉ tích lũy gradient; optimizer mới cập nhật tham số.
- Dùng `model.eval()` cho inference và `torch.no_grad()` khi không cần graph.
- Shape đúng không đảm bảo broadcasting đúng ý; luôn assert batch/feature dimensions.